<div style="background-color:#5A3516; color:#F3EEE6; padding:22px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h1 style="color:#F3EEE6; margin-bottom:0;"><b>Machine Learning II — Customer Segmentation</b></h1>
<h3 style="color:#D8C0B4; margin-top:6px;">Notebook 2 — Geographic Customer Analysis</h3>
<p style="color:#D8C0B4; font-size:15px; margin-top:14px;">This notebook explores customer latitude and longitude to understand spatial concentration, regional spread and whether geography should be used for profiling rather than based on distance clustering.</p>
</div>

<div style="background-color:#5A3516; color:#F3EEE6; padding:18px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin-top:0;"><b>Index</b></h2>
<p style="color:#D8C0B4; margin:6px 0 12px 0;">The geographic notebook is kept separate from the main preprocessing workflow so location can be analysed without forcing it into the clustering distance.</p>
<ol>
<li>Imports</li>
<li>Data loading</li>
<li>Basic geographic statistics</li>
<li>Customer geographic distribution</li>
<li>Geospatial mapping insights</li>
</ol>
</div>

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>1) Imports</b></h2>
</div>

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
import plotly.express as px
import warnings

import utils_eda as utils

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 8)
plt.rcParams["font.size"] = 12

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>2) Data loading</b></h2>
</div>

In [ ]:
DATA_DIR = Path("../datasets")
customer_info_clean = pd.read_csv(DATA_DIR / "info_clustering_unscaled.csv", index_col="customer_id")

print("Data loaded successfully")
print(f"Shape: {customer_info_clean.shape}")

In [ ]:
valid_coords = customer_info_clean[['latitude', 'longitude']].dropna()
print(f"Total customers: {len(customer_info_clean):,}")
print(f"Customers with coordinates: {len(valid_coords):,}")
print(f"Missing coordinates: {len(customer_info_clean) - len(valid_coords):,}")
print(f"Coverage: {len(valid_coords)/len(customer_info_clean)*100:.1f}%")

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>3) Basic geographic statistics</b></h2>
</div>

In [ ]:
print("\nLatitude Statistics")
print(f"Minimum: {customer_info_clean['latitude'].min():.4f}")
print(f"Maximum: {customer_info_clean['latitude'].max():.4f}")
print(f"Mean: {customer_info_clean['latitude'].mean():.4f}")
print("\nLongitude Statistics")
print(f"Minimum: {customer_info_clean['longitude'].min():.4f}")
print(f"Maximum: {customer_info_clean['longitude'].max():.4f}")
print(f"Mean: {customer_info_clean['longitude'].mean():.4f}")

In [ ]:
geo = customer_info_clean.loc[:, ['latitude','longitude']].copy()
geo.head()

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>4) Customer geographic distribution</b></h2>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(geo['longitude'], geo['latitude'], 
           s=2, alpha=0.3, color='blue')
ax.set_title('Clients Geographic Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Long')
ax.set_ylabel('Lat')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig = px.scatter_mapbox(geo, lat="latitude", lon="longitude",
                        zoom=6, height=400)
fig.update_layout(mapbox_style="open-street-map", margin = dict(l=0,r=0,t=0,b=0))
fig.show()

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">

To better understand the physical reach of our business, we visualize the customer base using a **Scatter Mapbox**. By mapping the geographical coordinates (`latitude` and `longitude`), we can identify regional density and potential market clusters.

### Insights from Geospatial Mapping
* **Regional Concentration:** The plot reveals a high density of customers within the **Lisbon Metropolitan Area**. This suggests that our primary market is centred around urban areas, which likely influences the high value of `distinct_stores_visited` due to store proximity.
* **Service Area Identification:** Visualizing the coordinates allows us to compare customer locations with store locations. This helps determine if spending habits (e.g., high spend on groceries) are correlated with living in specific affluent or easy access neighborhoods.
* **Clustering Context:** While geographic data is not always used directly as a clustering feature (to avoid biasing the model purely by location), it serves as a powerful **validation tool** to see if our behavioral clusters live in similar areas.

</div>

In [ ]:
valid_coords = geo[['latitude', 'longitude']].dropna()
center_lat = valid_coords['latitude'].mean()
center_lon = valid_coords['longitude'].mean()

m = folium.Map(
    location=[center_lat, center_lon], 
    zoom_start=11, 
    tiles='CartoDB positron'
)
cluster_layer = MarkerCluster(name="Customer Density Clusters").add_to(m)
for lat, lon in zip(valid_coords['latitude'], valid_coords['longitude']):
    folium.CircleMarker(
        location=[lat, lon],
        radius=3,
        color='#3186cc',
        fill=True,
        fill_color='#3186cc',
        fill_opacity=0.7
    ).add_to(cluster_layer)
m

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>5) Customer density map</b></h2>
</div>

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

hb = ax.hexbin(geo['longitude'], geo['latitude'], 
               gridsize=100, 
               cmap='YlOrRd', 
               bins='log',
               alpha=0.8)

ax.set_title('Customer Density Map (Log Scale)', fontsize=16, fontweight='bold')
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)

cbar = plt.colorbar(hb, ax=ax)
cbar.set_label('Number of Customers (log scale)', fontsize=12)

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<div style="background-color:#5A3516; color:#F3EEE6; padding:20px; border-radius:10px; border-left:8px solid #A8B7BA;">
<h2 style="color:#F3EEE6; margin:0;"><b>6) Hotspot profile and area near the university check</b></h2>
<p style="color:#D8C0B4; margin:6px 0 0 0;">At this stage the final clustering labels are not used yet. The goal is only to test whether the densest geographic area looks meaningfully different from the rest of the customer base, and whether it could later motivate a geographic profiling insight.</p>
</div>

In [ ]:
hotspot = utils.find_density_hotspot(customer_info_clean, bins=50)
hotspot_lat = hotspot["latitude"]
hotspot_lon = hotspot["longitude"]

print("Highest-density grid cell")
print(f"Latitude: {hotspot_lat:.4f}")
print(f"Longitude: {hotspot_lon:.4f}")
print(f"Customers in cell: {hotspot['customers_in_cell']:,}")

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
The densest point is calculated from a coordinate grid. This makes the hotspot analysis data driven and avoids defining the area from visual inspection alone.
</div>

In [ ]:
landmark_distances = utils.landmark_distances(hotspot_lat, hotspot_lon)
display(landmark_distances.round(3))

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
The hotspot is close to Cidade Universitaria and Entrecampos, and is also relatively close to NOVA IMS in Campolide. This makes the university/area near transport infrastructure explanation plausible, but it is not enough by itself to classify the customers as students.
</div>

In [ ]:
hotspot_customers, outside_hotspot = utils.get_hotspot_customers(
    customer_info_clean,
    hotspot_lat,
    hotspot_lon,
    radius=0.006,
)

print(f"Customers in hotspot area: {len(hotspot_customers):,}")
print(f"Share of dataset: {len(hotspot_customers) / len(customer_info_clean) * 100:.2f}%")

In [ ]:
profile_cols = [
    "customer_age",
    "education_level",
    "total_children",
    "percentage_of_products_bought_promotion",
    "log_total_spend",
    "lifetime_total_distinct_products",
    "distinct_stores_visited",
    "number_complaints",
    "tenure",
]

hotspot_profile = utils.compare_hotspot_profile(
    hotspot_customers,
    outside_hotspot,
    profile_cols,
)

important_differences = hotspot_profile.reindex(
    hotspot_profile["standardized_difference"].abs().sort_values(ascending=False).index
)

display(hotspot_profile.round(2))
display(important_differences.round(2).head(6))

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
The barplots below make the hotspot comparison easier to read visually. They compare the average behaviour of customers inside the dense area against customers outside it, using the same variables shown in the profile table.
</div>

In [ ]:
habit_cols = [
    "percentage_of_products_bought_promotion",
    "log_total_spend",
    "lifetime_total_distinct_products",
    "distinct_stores_visited",
    "number_complaints",
    "tenure",
    "total_children",
]

utils.plot_hotspot_comparison(
    hotspot_customers,
    outside_hotspot,
    habit_cols,
    title="Hotspot vs outside: behavioural profile",
)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
The hotspot shows a distinct profile even before using any clustering labels. The strongest differences are age, product diversity, number of complaints, store visits, total spend and promotion usage. This suggests that the dense area is not only a map artefact; it also corresponds to a younger and more sensitive to prices customer concentration.
</div>

In [ ]:
age_summary, age_distribution = utils.hotspot_age_tables(
    hotspot_customers,
    outside_hotspot,
)

print(f"Median age in hotspot: {hotspot_customers['customer_age'].median():.0f}")
print(f"Median age outside hotspot: {outside_hotspot['customer_age'].median():.0f}")

display(age_summary.round(2))
display(age_distribution.round(1))

In [ ]:
utils.plot_hotspot_age_distribution(
    hotspot_customers,
    outside_hotspot,
)

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
Age is the strongest signal supporting the area near the university hypothesis. The hotspot median age is much lower than the rest of the base, and the 25-34 band dominates the local profile. This is consistent with a younger urban population around university and transport infrastructure, although the data does not directly identify students.
</div>

<div style="background-color:#7E6A43; color:#F3EEE6; padding:15px; border-radius:10px;">
<b>Does this make sense as a cluster?</b> At this point, it should not be treated as one of the final customer segments because the segmentation model has not been fitted yet and geography is not part of the clustering distance. However, it does make sense as a <b>geographic hotspot</b>: the area is very dense, close to Cidade Universitaria and Entrecampos, and has a clearly younger, more oriented toward promotions and lower spending profile than the rest of the base. This can be kept as a profiling insight to revisit after clustering, rather than as a modelling feature.
</div>